# AutoGuard AI — Sprint 1 ML Methodology

This is the canonical reproducible pipeline. It splits the authoritative 10,000-row real source before creating any additional training data. Model selection uses Real Validation only; Real Test is evaluated once after the winner is frozen.


## 1. Configuration and source data

Authoritative source: `02_Data/raw/Car_Insurance_Claim.csv`. Target: `OUTCOME`; technical identifier: `ID`. All remaining source attributes contribute to duplicate-content fingerprints. Fixed random seed: 42.


In [1]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == '01_Notebook' else Path.cwd().resolve()
SCRIPT = ROOT / '02_Data' / 'generation' / 'build_training_data.py'
assert SCRIPT.exists(), SCRIPT
print('Repository root:', ROOT)


Repository root: C:\Users\97252\Desktop\ML CARS project\AutoGuard_AI_Sprint1


## 2. Execute the deterministic partition and training-only preparation

`build_training_data.py` uses `StratifiedGroupKFold` over full-record content fingerprints (excluding ID and target). This keeps exact duplicate source records in one real partition. It then bootstraps 40,000 additional rows from `real_train` only, with newly assigned generated IDs.


In [2]:
run = subprocess.run([sys.executable, str(SCRIPT)], cwd=ROOT, text=True, capture_output=True, check=True)
print(run.stdout)
if run.stderr: print(run.stderr)


SPRINT 1 COMPLETE
Partition rows: 6400 1599 2001
Winner selected on Real Validation: Logistic Regression
FINAL TEST RESULTS: {"accuracy": 0.8251, "precision": 0.7034, "recall": 0.764, "f1": 0.7324, "roc_auc": 0.8903, "confusion_matrix": [[1172, 202], [148, 479]], "n_evaluated": 2001}
SHA-256: 71d1474c36c4719a950e62659c34d28701c6ec66e072e01b18d6f67344932daf

C:\Users\97252\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\97252\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\97252\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.p

## 3. Data Leakage Validation

The following assertions were executed inside the preparation script for Real Train/Validation/Test and Training Pool/Validation/Test: ID overlap = 0 and content-fingerprint overlap = 0.


In [3]:
audit = json.loads((ROOT / '02_Data' / 'processed' / 'sprint1_audit.json').read_text(encoding='utf-8'))
audit['audits']


[{'pair': 'Real Train / Real Validation',
  'id_overlap': 0,
  'content_overlap': 0},
 {'pair': 'Real Train / Real Test', 'id_overlap': 0, 'content_overlap': 0},
 {'pair': 'Real Validation / Real Test',
  'id_overlap': 0,
  'content_overlap': 0},
 {'pair': 'Training Pool / Real Validation',
  'id_overlap': 0,
  'content_overlap': 0},
 {'pair': 'Training Pool / Real Test', 'id_overlap': 0, 'content_overlap': 0}]

## 4. Candidate training and validation-only model selection

Baseline, Logistic Regression, Random Forest, and Neural Networks A/B/C are fitted on the approved Training Pool only. The shared comparison table below is calculated on Real Validation only. The winner is selected by ROC-AUC (F1 only breaks a tie).


In [4]:
metadata = json.loads((ROOT / '03_Model' / 'model_v2_metadata.json').read_text(encoding='utf-8'))
metadata['candidate_validation_results'], metadata['model_type']


([{'accuracy': 0.8143,
   'precision': 0.6796,
   'recall': 0.7705,
   'f1': 0.7222,
   'roc_auc': 0.8876,
   'model': 'Logistic Regression',
   'training_time_sec': 0.275},
  {'accuracy': 0.818,
   'precision': 0.6855,
   'recall': 0.7745,
   'f1': 0.7273,
   'roc_auc': 0.8874,
   'model': 'Neural Network C',
   'training_time_sec': 17.572},
  {'accuracy': 0.8074,
   'precision': 0.6771,
   'recall': 0.7365,
   'f1': 0.7055,
   'roc_auc': 0.8805,
   'model': 'Neural Network B',
   'training_time_sec': 10.514},
  {'accuracy': 0.8124,
   'precision': 0.7152,
   'recall': 0.6667,
   'f1': 0.6901,
   'roc_auc': 0.8757,
   'model': 'Neural Network A',
   'training_time_sec': 1.175},
  {'accuracy': 0.7905,
   'precision': 0.6554,
   'recall': 0.6986,
   'f1': 0.6763,
   'roc_auc': 0.8317,
   'model': 'Random Forest',
   'training_time_sec': 1.704},
  {'accuracy': 0.6867,
   'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0,
   'roc_auc': 0.5,
   'model': 'Baseline',
   'training_time_sec': 0

## 5. Final refit and FINAL TEST RESULTS

After selection, the winning configuration is refit on Real Train + Real Validation + the training-only additional rows. No test record is included. The real Test set is then evaluated once, using the exported production artifact. No threshold optimization or calibration is performed in Sprint 1.


In [5]:
print('Selected model:', metadata['model_type'])
print('FINAL TEST RESULTS')
metadata['final_test_results']


Selected model: Logistic Regression
FINAL TEST RESULTS


{'accuracy': 0.8251,
 'precision': 0.7034,
 'recall': 0.764,
 'f1': 0.7324,
 'roc_auc': 0.8903,
 'confusion_matrix': [[1172, 202], [148, 479]],
 'n_evaluated': 2001}

## 6. Artifact integrity and serving compatibility

The script exports `03_Model/model_v2.pkl` and an identical notebook export, records their SHA-256 in metadata, and leaves the production backend and Gradio app loading the active `03_Model` artifact.


In [6]:
import hashlib
artifact = ROOT / '03_Model' / 'model_v2.pkl'
sha = hashlib.sha256(artifact.read_bytes()).hexdigest()
assert sha == metadata['artifact_sha256']
print('Artifact SHA-256 verified:', sha)


Artifact SHA-256 verified: 71d1474c36c4719a950e62659c34d28701c6ec66e072e01b18d6f67344932daf
